In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

print("PyTorch Version:", torch.__version__)
print("Torchvision Version:", torchvision.__version__)
print("Exercise 01 - MLP Environment Ready!")

In [2]:
import torchvision.datasets as datasets
import torchvision.transforms as transforms

# Convert images into PyTorch tensors
transform = transforms.ToTensor()

# Download Training Dataset
train_dataset = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

# Download Testing Dataset
test_dataset = datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

print("Training Images:", len(train_dataset))
print("Testing Images:", len(test_dataset))
print("CIFAR-10 Dataset Downloaded Successfully!")

In [3]:
import matplotlib.pyplot as plt

# CIFAR-10 class names
classes = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

# Display 10 images
fig, axes = plt.subplots(2, 5, figsize=(12, 6))

for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]

    # Convert tensor to image format
    image = image.permute(1, 2, 0)

    ax.imshow(image)
    ax.set_title(classes[label])
    ax.axis('off')

plt.tight_layout()
plt.show()

In [4]:
from torch.utils.data import random_split

# Reproducibility
generator = torch.Generator().manual_seed(42)

# Split training dataset
train_subset, val_subset = random_split(
    train_dataset,
    [45000, 5000],
    generator=generator
)

# Testing dataset remains separate
print("Training images:", len(train_subset))
print("Validation images:", len(val_subset))
print("Testing images:", len(test_dataset))

print("Dataset splitting completed successfully!")

In [5]:
from torch.utils.data import DataLoader

# Batch size
BATCH_SIZE = 64

# Training DataLoader
train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# Validation DataLoader
val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Testing DataLoader
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

print("DataLoaders created successfully!")

In [6]:
import torch.nn as nn

class MLP(nn.Module):

    def __init__(self):
        super(MLP, self).__init__()

        self.network = nn.Sequential(
            nn.Flatten(),

            nn.Linear(3072, 512),
            nn.ReLU(),

            nn.Linear(512, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.network(x)


# Create MLP model
model = MLP()

print(model)

In [7]:
# Get one batch of images
images, labels = next(iter(train_loader))

# Pass images through the model
outputs = model(images)

# Display shapes
print("Input shape:", images.shape)
print("Output shape:", outputs.shape)

# Count model parameters
total_params = sum(p.numel() for p in model.parameters())

print("Total model parameters:", total_params)

print("Model testing completed successfully!")

In [8]:
import torch.optim as optim

# Define loss function
criterion = nn.CrossEntropyLoss()

# Define optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss Function:", criterion)
print("Optimizer:", optimizer)
print("Loss Function and Optimizer configured successfully!")

In [9]:
def train_model(model, train_loader, criterion, optimizer):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item()

        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    accuracy = 100 * correct / total

    return epoch_loss, accuracy


print("Training function created successfully!")

In [10]:
def evaluate_model(model, data_loader, criterion):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in data_loader:

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(data_loader)
    accuracy = 100 * correct / total

    return epoch_loss, accuracy


print("Validation function created successfully!")

In [11]:
# Number of training epochs
EPOCHS = 10

# Store training history
train_losses = []
train_accuracies = []

val_losses = []
val_accuracies = []

# Start training
for epoch in range(EPOCHS):

    train_loss, train_acc = train_model(
        model, train_loader, criterion, optimizer
    )

    val_loss, val_acc = evaluate_model(
        model, val_loader, criterion
    )

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print(f"Training Loss: {train_loss:.4f}")
    print(f"Training Accuracy: {train_acc:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.2f}%")
    print("-" * 40)

print("MLP Training Completed!")

In [12]:
# Plot Training and Validation Accuracy
plt.figure(figsize=(8, 5))

plt.plot(train_accuracies, label='Training Accuracy')
plt.plot(val_accuracies, label='Validation Accuracy')

plt.title('MLP - Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)
plt.show()


# Plot Training and Validation Loss
plt.figure(figsize=(8, 5))

plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')

plt.title('MLP - Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [13]:
# Evaluate model on unseen test data
test_loss, test_accuracy = evaluate_model(
    model,
    test_loader,
    criterion
)

print("Final MLP Test Results")
print("----------------------")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")
print("Evaluation completed successfully!")

In [14]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Get predictions from test dataset
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_predictions.extend(predicted.numpy())
        all_labels.extend(labels.numpy())

# CIFAR-10 classes
classes = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

# Classification Report
print(classification_report(
    all_labels,
    all_predictions,
    target_names=classes
))

In [15]:
# Create confusion matrix
cm = confusion_matrix(all_labels, all_predictions)

# Display confusion matrix
plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=classes,
    yticklabels=classes
)

plt.title('MLP - Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.show()

In [16]:
import os

# Create models folder
os.makedirs("models", exist_ok=True)

# Save trained model
torch.save(
    model.state_dict(),
    "models/mlp_cifar10.pth"
)

print("MLP model saved successfully!")
print("Saved location: models/mlp_cifar10.pth")

In [17]:
import os

model_path = "models/mlp_cifar10.pth"

if os.path.exists(model_path):
    print("Model file exists!")
    print("File size:", round(os.path.getsize(model_path) / 1024, 2), "KB")
else:
    print("Model file not found!")

In [18]:
print("================================")
print("EXERCISE 01 - MLP FINAL SUMMARY")
print("================================")

print("Dataset: CIFAR-10")
print("Training Images:", len(train_subset))
print("Validation Images:", len(val_subset))
print("Testing Images:", len(test_dataset))

print("Model: Multi-Layer Perceptron")
print("Hidden Layers: 512, 256")
print("Output Classes: 10")
print("Optimizer: Adam")
print("Loss Function: CrossEntropyLoss")
print("Epochs:", EPOCHS)

print("--------------------------------")
print(f"Final Test Loss: {test_loss:.4f}")
print(f"Final Test Accuracy: {test_accuracy:.2f}%")
print("--------------------------------")

print("Exercise 01 completed!")